
# What this does
1. Sets up subject-specific output folders.
2. Loads master **stimulus pools** (faces/places/fruits) for both *main-task* and *novel* images.
3. Builds a **balanced condition skeleton** across operations, probe types, and ordered category pairs.
4. Pseudorandomizes trials **within each run** (6 runs × 24 trials = 144) to avoid 4-in-a-row operation streaks.
5. Assigns concrete **image files** per trial (left, right, replacement, probe) with fairness constraints (limits repeated probing of the same image and avoids in-run duplicates).
6. **Retries** building if constraints are too tight; **falls back** to a premade list if needed.
7. Saves the final design matrix to `subject lists/sub-<id>/main_task/main_stim_list.csv`.



# Imports
* Don't see a need to modify any of this 

importing the packages. random helps pull random numbers, pandas is data analytics package, numpy is an array processing/computing, os helps interact with the operating system (ex. making a file), colletions helps store data, Counter lets you count objects, copy can help copy objects, itertools lets you get the permutations (different combinations of outcomes)

In [ ]:

#### LOAD NECESSARY PACKAGES ####
import random
import pandas as pd
import numpy as np
import os
from collections import Counter
import copy
from itertools import permutations



# Participant & Working Directory
* Don't see a need to modify any of this


This basicallys sets each participant a unique number so that participant data doesn't overlap. Participant_number is just the subject’s name tag we stick into folder names (like sub-test) so each person’s files stay separate. _thisDir is the “home base” folder where the subjects specific data in its own folder gets stored. To determine where to read and write those files, the code sets _thisDir to the directory that contains the script using _file_; if _file_ is unavailable—as in notebooks or interactive consoles, it falls back to the current working directory.



In [ ]:

# Set participant number
# Need this to create a folder with each subject's design matrix
participant_number = "test"  # In PsychoPy: expInfo['participant']

# Set directory path robustly
try:
    _thisDir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    print("__file__ does not exist in this environment; using current working directory")
    _thisDir = os.getcwd()

_thisDir


# Create Subject / Phase Folders
* Don't see a need to modify any of this 

Basically makes a bunch of folders

Creates the subject-specific output directory and a `main_task` subfolder to hold the generated CSV.

Creates <project>/subject lists/sub-<participant>/main_task. If folders already exist, it logs that instead. It basically prevents a script error if the folder already exists. 


In [ ]:

# Create a subject-specific folder to save all lists
list_foldername = os.path.join(_thisDir, "subject lists", f"sub-{participant_number}")
phase_subFolder = "main_task"

# Create the subject folder if it doesn't exist
if not os.path.exists(list_foldername):
    os.makedirs(list_foldername, exist_ok=True)
    print(f"List folder successfully created for sub-{participant_number}")
else:
    print(f"List folder already exists for sub-{participant_number}...")

# Create the phase subfolder within the subject's folder
phase_folder_path = os.path.join(list_foldername, phase_subFolder)
if not os.path.exists(phase_folder_path):
    os.makedirs(phase_folder_path, exist_ok=True)
    print(f"Created phase folder: {phase_subFolder}")
else:
    print(f"Phase folder '{phase_subFolder}' already exists...")



# Setup Stimulus Master Lists
* Don't see a need to modify any of this
  
Loading the big lists of pictures (faces, places, fruits) that the task can use—both regular ones and “novel” (never-seen-before) ones.
- Seeds the RNG (no fixed seed → different shuffles each run).
- Loads main-task and novel stimulus lists for `faces`, `places`, `fruits` from CSV files.
- Each CSV is assumed to have a single column containing filenames (one per row).

Loads 6 CSVs into lists: 3 main_task pools and 3 novel pools for categories faces/places/fruits. Each CSV’s first column is read into a Python list.

Separate main vs novel pools let you create “novel” probes that were never shown during encoding.


In [ ]:
#### SETUP STIMULUS MASTER LISTS ####
# Initialize RNG (no fixed seed -> different randomization each run)
random.seed()

# Build absolute paths to stimulus CSVs
faces_mainTask_list_infile = os.path.join(_thisDir, "stimuli", "csvs", "main_task", "faces_mainTask.csv")
places_mainTask_list_infile = os.path.join(_thisDir, "stimuli", "csvs", "main_task", "places_mainTask.csv")
fruits_mainTask_list_infile = os.path.join(_thisDir, "stimuli", "csvs", "main_task", "fruits_mainTask.csv")

faces_novel_list_infile  = os.path.join(_thisDir, "stimuli", "csvs", "main_task", "faces_novel.csv")
places_novel_list_infile = os.path.join(_thisDir, "stimuli", "csvs", "main_task", "places_novel.csv")
fruits_novel_list_infile = os.path.join(_thisDir, "stimuli", "csvs", "main_task", "fruits_novel.csv")

# Load first column of each CSV as a list
# NOTE: These files must exist in your project structure.
faces_mainTask_list  = pd.read_csv(faces_mainTask_list_infile,  header=None).loc[:,0].tolist()   # ~20 items
places_mainTask_list = pd.read_csv(places_mainTask_list_infile, header=None).loc[:,0].tolist()   # ~20 items
fruits_mainTask_list = pd.read_csv(fruits_mainTask_list_infile, header=None).loc[:,0].tolist()   # ~20 items

faces_novel_list  = pd.read_csv(faces_novel_list_infile,  header=None).loc[:,0].tolist()         # novel pool
places_novel_list = pd.read_csv(places_novel_list_infile, header=None).loc[:,0].tolist()         # novel pool
fruits_novel_list = pd.read_csv(fruits_novel_list_infile, header=None).loc[:,0].tolist()         # novel pool

# Quick peek at counts (will error if files are missing)
len(faces_mainTask_list), len(places_mainTask_list), len(fruits_mainTask_list), len(faces_novel_list), len(places_novel_list), len(fruits_novel_list)



# Experiment Constants & Factors
* for my experiment, I will need to change the operations to just maintain and suppress, with the same categories (faces/places/fruits), probe_types (0-3), and cued position of image (right or left)


Defines total trials, runs, and factor levels (operations, categories, probe types).

Sets global schedule: num_runs=6, run_len=24, total_trials=144.
Names your factors: operations (maintain/suppress/replace), categories (faces/places/fruits), probe_types (0–3), the cued position of image can be left or right.

These constants define the backbone of the design; many balance checks use them (e.g., 24 trials / 6 ordered pairs = 4 per pair in a tray).


In [ ]:
#### CREATE MAIN TASK LISTS ####
# Define some experiment variables
total_trials = 192    # number of trials
run_len = 48          # trials per run
num_runs = 4          # number of runs
num_oper_per_run = run_len // 2    # 2 operations -> 48 // 2 = 24 per operation per run

## Create conditions structure
# Factor levels
cue_pos    = ["left", "right"]
operations = ["maintain", "suppress"]
categories = ["faces", "places", "fruits"]
probe_types = range(4)  # "cued", "ucued", "novel_samecatcued", "novel_samecatuncued"



# Build the Balanced Condition Skeleton
* because my experiment does not have a replace conidtion, I will need to remove replace. Each operation has 4 probe types × 6 pairs × 3 repeats = 72 rows in total, and repeating the 48-trial block three times yields 144 trials overall. combined_conds is (144, 4): a flat list of planned trials ready for assigning runs and pseudorandomizing.


 
  Why 4 columns
Each row/trial stores exactly 4 things:
encode_1_cat
encode_2_cat
operation
probe_type (the index you’ll later map to cued/uncued/novel_samecat)
2 operations: maintain, suppress
4 probe indices: 0, 1, 2, 3
(later mapped to cued/cued/uncued/novel_samecat → 50/25/25)
3 categories: faces, places, fruits
Ordered pairs of different categories: permutations(3, 2) = 6
(e.g., faces→places, places→faces, faces→fruits, etc.)

2 (ops)×4 (probe ids)×6 (pairs)=48


Making the template of trials without picking the actual pictures yet—just the plan (e.g., “faces then places, with a suppress operation, probe type #2”).
- Generate all combinations of `operation × probe_type × ordered category pairs (A→B, B→A)`.
- Stack into three 24-trial blocks (one per operation).

Guarantees full factorial coverage and intra-operation balance across ordered pairs and probe slots before you start shuffling.

Each operation has 4 probe types × 6 pairs = 24 rows → a 24×4 tray.
combined_conds is (3, 24, 4) (3 trays).


In [ ]:
# Create the conditions list
conditions = [[(cat1, cat2, oper, probe)
               for cat1, cat2 in permutations(categories, 2)]
              for oper in operations
              for probe in probe_types]
conditions = np.array(conditions)     # shape = (8 blocks, 6 rows, 4 cols) with 2 ops & 4 probe ids

# Concatenate within operation to create 2 arrays of 24 (shape=[2, 24, 4])
combined_conds = np.array([
    np.vstack(conditions[:4]),     # maintain
    np.vstack(conditions[4:8])    # suppress
])

combined_conds.shape  # (2, 24, 4)
print(combined_conds)
tray_len = combined_conds.shape[1]  # = 24


# Interleave & Split into Three 24-Trial Sets + Sanity Checks

* edited: 2 → number of operations (maintain, suppress)
24 → trials per operation (6 category pairs × 4 probe types = 24)
4 → the 4 fields in each tuple (encode_1_cat, encode_2_cat, operation, probe_type)

Mixing the template in a smart pattern so operations and category-pairs are spread out, then double-checking the counts are fair.
Prevents lopsided sets (e.g., too many “faces→places” in one chunk).
- Create a 3×24 index (`cond_inds`) to interleave operation blocks.
- Randomly circular-shift columns to avoid deterministic order.
- Extract three balanced 24-trial trays and assert balance over category pairs and operations.

Builds a 3×24 index matrix cond_inds that cycles 0/1/2 with a phase shift, then randomly rolls columns to avoid a deterministic pattern.
Uses that matrix to pull out three balanced 24-row sets (run_1_conds, run_2_conds, run_3_conds) from combined_conds.
Asserts:
each of the 6 ordered category pairs appears exactly 4 times (24/6), and
each of the 3 operations appears exactly 8 times (24/3) within a tray.
You want each 24-trial tray to be internally balanced across category pairs and operations, so any later shuffling won’t break global balance.


In [ ]:
# Create indices for the two conditions
cond_inds = np.full((len(operations), tray_len), np.nan)
for i in range(tray_len):
    cond_inds[:, i] = np.roll(np.arange(2), (i % 2))

cond_inds = np.roll(cond_inds, np.random.randint(tray_len), axis=1)

# Use boolean indexing to separate out the three trays
run_1_conds = combined_conds[cond_inds == 0]
run_2_conds = combined_conds[cond_inds == 1]

# Sanity checks for balance in each 24-trial tray
for run, conds in enumerate([run_1_conds, run_2_conds], start=1):
    # Check the categories (ordered pairs)
    c = Counter([(r[0], r[1]) for r in conds])
    assert all([cat in c.keys() for cat in permutations(categories, 2)]),         "Missing category pair(s)"
    assert all([v == tray_len // len(list(permutations(categories, 2))) for v in c.values()]),         f"Imbalance in category pairs within tray {run}"

    # Check the operations
    c = Counter(conds[:, 2])
    assert all([op in c.keys() for op in operations]), "Missing operation(s)"
    assert all([v == tray_len // len(operations) for v in c.values()]),         f"Imbalance in operations within tray {run}"

"Sanity checks passed."



# Build 144-Row Table (First Half + Mirrored Second Half) & Run Labels
* need to change, doesn't have to be precisely mirrored, make 96 trials fit into 144= probelm work out 

Solution: so there are 48 set of unique conditions in 144 trials, therefore each unqiue set of conditions shows 3 times in the entire experiment. Cue position (right or left) can not show evenly amongst those 3 times of unique conditions (3 not divisible by 2). An option is reducing the total amount of trials to 96 to evenly balance and mirror cue position. This can separate it by 48 mirrored to 48 for the two operations, two cue positions, and 4 probe types OR *what was done* for each specific combination of encode_1_cat × encode_2_cat × operation (for example, faces→places with “maintain”), it finds the 12 trials in that group and gives exactly 6 of them cue_position = 0 (left) and 6 of them cue_position = 1 (right) in random order. Doing this for every group guarantees two things: overall across the whole experiment there are 72 left and 72 right cues, and within every category-pair × operation group, left and right are perfectly balanced. The only place it’s not perfectly balanced is at level of each individual probe_type
  
Taking 72 trials for the “first half,” then cloning them for a “second half” but flipping which side is cued (left/right) to balance things. Also tagging which run (1–6) each trial belongs to.
Guarantees left/right cueing is perfectly even, and we end up with 6 equal runs.
- Shuffle the three trays to form the first half (72 trials).
- Assign 50/50 cue sides at random.
- Create the second half by flipping cue side and reordering blocks.
- Label `run_num` (1–6).

Randomly orders the three 24-row trays and stacks them into a 72-row first half.
Creates a balanced, randomized cue-side vector (36 left, 36 right).
Makes a second half by copying the first half and flipping cue side (left↔right), then reorders 24-row blocks before appending to yield 144 rows.
Adds run_num = 1..6 (24 trials per run).
The flip ensures perfect counterbalancing of cue side across identical condition rows.
The block reordering avoids having halves be trivially mirrored in sequence.




In [ ]:
# --- BUILD 144 TRIALS FROM THE 48 UNIQUE CONDITIONS (OPTION A) ---

# combined_conds should be shape (2, 24, 4) for 2 ops × 24 conds each = 48 conditions total
base_df = pd.DataFrame(
    combined_conds.reshape(-1, 4),   # (48, 4)
    columns=column_order
)

# Repeat 4 times to get 192 trials
conditions_df = pd.concat([base_df] * 4, ignore_index=True)   # (192, 4)
assert len(conditions_df) == total_trials


# --- ASSIGN CUE POSITION, BALANCED WITHIN (cat1, cat2, op) GROUPS ---

conditions_df["cue_position"] = -1
group_cols = ["encode_1_cat", "encode_2_cat", "operation"]

for _, idx in conditions_df.groupby(group_cols).groups.items():
    idx = list(idx)
    n = len(idx)             # now should be 16 trials per (pair × operation)
    assert n % 2 == 0
    cue_vals = np.array([0] * (n // 2) + [1] * (n // 2))  # 8 left, 8 right
    np.random.shuffle(cue_vals)
    conditions_df.loc[idx, "cue_position"] = cue_vals

# --- RANDOMIZE TRIAL ORDER ---

conditions_df = conditions_df.sample(frac=1).reset_index(drop=True)

# --- ADD RUN NUMBERS: 4 runs × 48 trials ---

conditions_df["run_num"] = np.repeat(np.arange(1, num_runs + 1), run_len)

conditions_df.head()



# Pseudorandomize Within Runs (No 4 Identical Ops in a Row)
* Don't see a need to modify any of this 

Shuffling the 24 trials inside each run until you never get 4 of the same operation back-to-back.
For each run, shuffles its 24 rows until there is no sequence of 4 identical operation values (checked with chained shift comparisons).
Concatenates valid runs to form mainTask_df.

Long streaks can bias behavior and feel unnatural to participants. 

In [ ]:

## Pseudorandomize trials within runs and append to mainTask dataframe
mainTask_df = pd.DataFrame()

for i in range(1, num_runs+1):
    run_df = conditions_df[conditions_df["run_num"] == i]
    wm_valid_sequence = False
    while not wm_valid_sequence:
        run_df = run_df.sample(frac=1).reset_index(drop=True)
        wm_valid_sequence = not any(
            (run_df['operation'] == run_df['operation'].shift(1)) &
            (run_df['operation'] == run_df['operation'].shift(2)) &
            (run_df['operation'] == run_df['operation'].shift(3))
        )
    mainTask_df = pd.concat((mainTask_df, run_df), ignore_index=True)

mainTask_df.head()



# Recodes & Derived Columns
* Only changed the probe subtype mapping to fit the 50% novel, 25$ cued, 25% uncued  

Modified by getting rid of portion that gives back N/A for replacement trials and took away the probe subtupe that includes replace. 

Turning numbers into readable labels (like “left”/“right”): Converts cue_position from 0/1 to "left"/"right".
Maps numeric probe_type (0–3) to operation-specific probe_subtype (e.g., maintain: ["cued","uncued","novel","novel"]).
Computes replace_cat = the missing category among the three for replace trials (set difference).
Adds trial_num (1–24 within each run), balanced jitter per run ([3,4,5] repeated and shuffled), and a rest_trigger (1 on each run’s last trial, except forced 0 on the final trial of the task).


PsychoPy usually wants human-readable columns and explicit trial metadata.
probe_subtype drives how probes are chosen later.
replace_cat tells you which category the replacement should come from (the category not encoded).


In [ ]:

## recode the conditions into variable names used in PsychoPy
mainTask_df["cue_position"] = ["right" if cp else "left" for cp in mainTask_df.cue_position]

# Convert the probe_type to int
mainTask_df["probe_type"] = mainTask_df["probe_type"].astype(int)

# Probe subtype mapping per operation
maintain_probes = ["cued", "uncued", "novel_samecatuncued", "novel_samecatcued"]      # CHANGED
suppress_probes = ["cued", "uncued", "novel_samecatuncued", "novel_samecatcued"]      # CHANGED

probe_subtype = []
for _, r in mainTask_df.iterrows():
    if r.operation == "maintain":
        probe_subtype.append(maintain_probes[r.probe_type])
    elif r.operation == "suppress":
        probe_subtype.append(suppress_probes[r.probe_type])
mainTask_df["probe_subtype"] = probe_subtype

# Trial number within run 
mainTask_df["trial_num"] = list(range(1, run_len + 1)) * num_runs

# Jitter: per run, balanced [3,4,5] shuffled
jitter = []
for _ in range(num_runs):
    jitter += np.random.permutation([3, 4, 5] * int(run_len / 3)).tolist()
mainTask_df["jitter"] = jitter

# Rest trigger: 1 on last trial of each run, except last trial of the whole task forced to 0
mainTask_df["rest_trigger"] = [
    int(r.trial_num == run_len) if i != (total_trials - 1) else 0
    for i, r in mainTask_df.iterrows()
]

mainTask_df.head()



# Helper Functions for Stimulus Selection
A picker that chooses a specific image file for each slot (left/right/replacement) while avoiding repeats in the same run and spreading “probe duty” fairly across images.
A checker that confirms each trial actually has all its images filled in.
- `select_stim(...)`: Picks an image from the correct pool (operation × cue-status × category), balancing probe usage and avoiding in-run repeats.
- `check_trials(...)`: Verifies every trial has `left`, `right`, `replacement`, and `probe` images.

What (select_stim):
Inputs: global probe-usage counter, list of images already used in the current run (crs_list), nested stim pools (stimsdict), flags about whether this slot is the probed one, and the operation/cue-status/category for this slot.

Build a Counter over all images still present in the nested pools (measures availability across pools).
Filter to the specific pool required by this trial (op × cue-status × category).
For probed slots, sort candidates by fewest prior probes (from probecounter) to spread probe duty; for non-probed slots, reverse=True pushes more-probed items first so they get consumed as encoding items (reducing chance they’ll be probed again later).
Ensure no within-run repeats using current_run_stims.
Remove the chosen item from the pool so it can’t be reused incorrectly.

This balances probe load across images (a common fairness constraint) and avoids local repeats within a run.

What (check_trials):
Verifies each trial dict has non-None entries for left, right, replacement, and probe.
Stops partially built runs from being accepted.


In [ ]:
def select_stim(probecounter, crs_list, stimsdict, useprobe=True,
                oper=None, cuestatus=None, imgcat=None, reverse=False,):
    """
    Select a concrete stimulus filename from the available pool, honoring:
    - balancing of how often items have been probed (probecounter),
    - avoiding reuse within the current run (crs_list),
    - pulling from the correct pool stimsdict[operation][cue_status][category],
    - optionally prioritizing least-probed items for the probed position (useprobe=True).
    If reverse=True, invert ordering to consume more-used items for non-probed slots.
    """
    if oper is not None and cuestatus is not None and imgcat is not None:
        pool = stimsdict[oper][cuestatus][imgcat]
        if len(pool) < 5:  # threshold; tweak if you want
            if imgcat == "faces":
                refill = np.random.permutation(faces_mainTask_list).tolist()
            elif imgcat == "places":
                refill = np.random.permutation(places_mainTask_list).tolist()
            elif imgcat == "fruits":
                refill = np.random.permutation(fruits_mainTask_list).tolist()
            else:
                refill = []

            # add only items not already in the pool to avoid giant duplicates
            existing = set(pool)
            pool.extend([s for s in refill if s not in existing])
            stimsdict[oper][cuestatus][imgcat] = pool
            
    counter = Counter()
    for o in stimsdict:
        for p in stimsdict[o]:
            for c in stimsdict[o][p]:
                counter.update(stimsdict[o][p][c])

   viable_list = [sv for sv in counter.most_common() if sv[0] in stimsdict[oper][cuestatus][imgcat]]

    for value in np.unique([v for _, v in viable_list]):
        stims = [s[0] for s in viable_list if s[1] == value]

        if useprobe:
            sortedstims = [k for k in sorted(probecounter.items(), key=lambda x:x[1]) if k[0] in stims]
            if reverse:
                sortedstims.reverse()
        else:
            sortedstims = np.random.permutation(stims)

        if useprobe:
            for val in np.unique([k[1] for k in sortedstims]):
                substims = np.random.permutation([k[0] for k in sortedstims if k[1] == val])
                for s in substims:
                    if s not in crs_list:
                        stimsdict[oper][cuestatus][imgcat].remove(s)
                        crs_list.append(s)
                        return s
        else:
            for s in sortedstims:
                if s not in crs_list:
                    stimsdict[oper][cuestatus][imgcat].remove(s)
                    crs_list.append(s)
                    return s
    return None

def check_trials(imglist: list, numtrials: int):
    """
    Ensure each trial in imglist has non-None images for:
    'left', 'right', and 'probe'.
    """
    return (len([i["left"]        for i in imglist if i["left"]        is not None]) == numtrials and
            len([i["right"]       for i in imglist if i["right"]       is not None]) == numtrials and
            len([i["probe"]       for i in imglist if i["probe"]       is not None]) == numtrials)

# Initiate control flags and operation cue images
check = False
retries = 0
opcue_dict = {
    "maintain": "stimuli/cues/maintain_cue_image.png",
    "suppress": "stimuli/cues/suppress_cue_image.png",
}

# ------------ BUILD IMAGE LISTS SAFELY (NO MISSING IMAGES) WITH FALLBACK ------------

build_succeeded = False
max_retries = 20

for attempt in range(max_retries):
    # Assign stimulus pools (fresh each attempt)
    stims_dict = {}
    for oper in ["maintain", "suppress"]:
        stims_dict[oper] = {}
        for probe in ["cued", "uncued"]:
            stims_dict[oper][probe] = {
                "faces":  np.random.permutation(faces_mainTask_list).tolist(),
                "places": np.random.permutation(places_mainTask_list).tolist(),
                "fruits": np.random.permutation(fruits_mainTask_list).tolist(),
            }

    # Novel probe pools
    novel_stims_dict = {
        "faces":  np.random.permutation(faces_novel_list).tolist(),
        "places": np.random.permutation(places_novel_list).tolist(),
        "fruits": np.random.permutation(fruits_novel_list).tolist(),
    }

    # Counter: how many times each main-task image has been used as a probe
    timesprobed_counter = Counter(faces_mainTask_list + places_mainTask_list + fruits_mainTask_list)
    for entry in timesprobed_counter:
        timesprobed_counter[entry] = 0

    image_list = []   # all trials across runs
    build_ok = True   # will flip to False if any run fails

    # -------- per-run build --------
    for run in range(1, num_runs + 1):
        df = mainTask_df[mainTask_df["run_num"] == run]
        run_check = False
        t = 0
        useprobe = True  # prefer strict balancing at first

        while not run_check and t < 10:
            random.seed()
            run_image_list = []      # per-trial images for this run
            current_run_stims = []   # track images used in this run (avoid repeats)

            # local copies so failures don't corrupt global state
            run_timesprobed_counter = copy.deepcopy(timesprobed_counter)
            run_stims_dict        = copy.deepcopy(stims_dict)
            run_novel_stims_dict  = copy.deepcopy(novel_stims_dict)

            # iterate trials for this run
            for _, r in df.iterrows():
                # decide probe position
                if r.probe_subtype == "cued":
                    probe_pos = r.cue_position
                elif r.probe_subtype == "uncued":
                    probe_pos = "left" if r.cue_position == "right" else "right"
                else:
                    probe_pos = None  # novel

                images_dict = {}
                for imgnum in ["left", "right"]:
                    imgcat = r.encode_1_cat if imgnum == "left" else r.encode_2_cat
                    cue_status = ("uncued", "cued")[r.cue_position == imgnum]

                    if imgnum == probe_pos:
                        img = select_stim(
                            run_timesprobed_counter,
                            current_run_stims,
                            run_stims_dict,
                            useprobe=useprobe,
                            oper=r.operation,cuestatus=cue_status,
                            imgcat=imgcat,
                            reverse=False
                        )
                    else:
                        img = select_stim(
                            run_timesprobed_counter,
                            current_run_stims,
                            run_stims_dict,
                            useprobe=useprobe,
                            oper=r.operation,
                            cuestatus=cue_status,
                            imgcat=imgcat,
                            reverse=True
                        )
                    images_dict[imgnum] = img

                # probe image
                if probe_pos is not None:
                    probe_img = images_dict[probe_pos]
                    run_timesprobed_counter[probe_img] += 1
                else:
                    probe_cat = np.random.choice([r.encode_1_cat, r.encode_2_cat])
                    probe_img = run_novel_stims_dict[probe_cat].pop()

                images_dict["probe"] = probe_img
                run_image_list.append(images_dict)

            # check this run: no None anywhere
            run_check = check_trials(run_image_list, run_len)
            t += 1
            if t > 5:
                useprobe = False  # relax balancing
 if not run_check:
            # this run never succeeded -> abort this whole build attempt
            build_ok = False
            break

        # commit run to global state
        image_list += run_image_list
        timesprobed_counter = run_timesprobed_counter
        stims_dict          = run_stims_dict
        novel_stims_dict    = run_novel_stims_dict

    # after all runs for this attempt
    if build_ok and len(image_list) == total_trials and check_trials(image_list, total_trials):
        print(f"completed build on attempt {attempt+1}, now checking... success!")
        build_succeeded = True
        break
    else:
        print(f"DNF, restarting... (build_ok={build_ok}, n_trials={len(image_list)})")

# ------------ USE BUILT LIST *OR* FALL BACK TO PREMADE ------------

if build_succeeded:
    # Start from conditions_df and add images
# Attach images to the existing mainTask_df (which already has trial_num, jitter, rest_trigger)
    mainTask_df["encode_1_img"] = [i["left"]  for i in image_list]
    mainTask_df["encode_2_img"] = [i["right"] for i in image_list]
    mainTask_df["probe_img"]    = [i["probe"] for i in image_list]
    print(mainTask_df["operation"].value_counts())


    # HARD SANITY CHECK: no missing images in any encoded/probe slot
    if mainTask_df[["encode_1_img", "encode_2_img", "probe_img"]].isnull().any().any():
        raise ValueError("Stim list has missing images in encode_1_img / encode_2_img / probe_img – this should not happen.")
else:
    # FALLBACK: use an existing premade list
    premade_infile = os.path.join(
        _thisDir,
        "stimuli", "csvs", "maintask_stimlists",
        f"main_stim_list_{random.randint(0,9)}.csv" 
    )
    print(f"Using pre-made stim list {os.path.split(premade_infile)[1]} after build failure.")
    mainTask_df = pd.read_csv(premade_infile)

# Make sure run_num and trial_num exist (in case premade file doesn't have them)
if "run_num" not in mainTask_df.columns:
    mainTask_df["run_num"] = np.repeat(np.arange(1, num_runs + 1), run_len)

if "trial_num" not in mainTask_df.columns:
    mainTask_df["trial_num"] = list(range(1, total_trials + 1))

# ------------ SAVE SUBJECT-SPECIFIC LIST ------------

# Convert numeric cue_position (0/1) to strings for PsychoPy
mainTask_df["cue_position"] = mainTask_df["cue_position"].replace({0: "left", 1: "right"})

mainTask_df_infile = os.path.join(list_foldername, "main_task", "main_stim_list.csv")
mainTask_df.to_csv(mainTask_df_infile, index=False)
stim_list = mainTask_df_infile



# Main Stimulus Assignment Loop (with Retries & Relaxation)
The engine that, run by run, fills in the actual pictures for each trial
- Builds nested dictionaries of available images per (operation × cue-status × category) and separate novel pools.
- Tracks how often each main-task image has been probed; prefers least-probed items for the probed slot.
- Avoids reusing the same image within a run.
- Retries with relaxed balancing if an attempt fails; after many retries, will fallback to a premade list.

For each attempt:
Initializes stims_dict as nested pools per operation × cue-status (cued/uncued/replacement) × category.
Note: replacement pool only exists for replace trials; non-replace trials show a cue image in the replacement slot.
Initializes novel_stims_dict (per category) for “novel” probes.
Initializes timesprobed_counter to 0 for every main-task image.
For each run:
Attempts to build the run up to ~10 times (t counter).
First 5 attempts use useprobe=True (strict probe balancing); later attempts drop to False (more random) to increase solvability.
For each trial in the run:
Decides which slot will be probed (probe_pos) from probe_subtype and cue side.
Chooses left/right/replacement images using select_stim:
Probed slot: prefer least-probed items (to spread probe duty).
Non-probed slots: prefer the more-probed ones (so they get used up as encodes rather than being probed again).
Sets probe_img:
If not “novel,” the probe is the image at probe_pos; increments run_timesprobed_counter.
If “novel,” randomly pick one encoded category and pop from run_novel_stims_dict for that category.
Validates the run with check_trials. If it fails too many times: break and restart the run.
On success, commit the run’s counters and pools back to the global copies.
After all runs, validate the full 144-trial image list; if OK, check=True; else restart the whole build.
Why:
These nested retries make the builder robust under competing constraints (balance, uniqueness within runs, limited pool sizes).
The “relax then retry” approach trades optimal probe balancing for feasibility when needed.



In [ ]:

# Initiate control flags and operation cue images
check = False
retries = 0
opcue_dict = {
    "maintain": "stimuli/cues/maintain_cue_image.png",
    "suppress": "stimuli/cues/suppress_cue_image.png",
}

# ------------ BUILD IMAGE LISTS SAFELY (NO MISSING IMAGES) ------------

check = False
retries = 0
max_retries = 20

while not check and retries < max_retries:
    # Assign stimulus pools (fresh each attempt)
    stims_dict = {}
    for oper in ["maintain", "suppress"]:
        stims_dict[oper] = {}
        for probe in ["cued", "uncued"]:
            stims_dict[oper][probe] = {
                "faces":  np.random.permutation(faces_mainTask_list).tolist(),
                "places": np.random.permutation(places_mainTask_list).tolist(),
                "fruits": np.random.permutation(fruits_mainTask_list).tolist(),
            }

    # Novel probe pools
    novel_stims_dict = {
        "faces":  np.random.permutation(faces_novel_list).tolist(),
        "places": np.random.permutation(places_novel_list).tolist(),
        "fruits": np.random.permutation(fruits_novel_list).tolist(),
    }

    # Counter: how many times each main-task image has been used as a probe
    timesprobed_counter = Counter(faces_mainTask_list + places_mainTask_list + fruits_mainTask_list)
    for entry in timesprobed_counter:
        timesprobed_counter[entry] = 0

    image_list = []   # all trials across runs
    build_ok = True   # will flip to False if any run fails

    # -------- per-run build --------
    for run in range(1, num_runs + 1):
        df = mainTask_df[mainTask_df["run_num"] == run]
        run_check = False
        t = 0
        useprobe = True  # prefer strict balancing at first

        while not run_check and t < 10:
            random.seed()
            run_image_list = []      # per-trial images for this run
            current_run_stims = []   # track images used in this run (avoid repeats)

            # local copies so failures don't corrupt global state
            run_timesprobed_counter = copy.deepcopy(timesprobed_counter)
            run_stims_dict        = copy.deepcopy(stims_dict)
            run_novel_stims_dict  = copy.deepcopy(novel_stims_dict)

            # iterate trials for this run
            for _, r in df.iterrows():
                # decide probe position
                if r.probe_subtype == "cued":
                    probe_pos = r.cue_position
                elif r.probe_subtype == "uncued":
                    probe_pos = "left" if r.cue_position == "right" else "right"
                else:
                    probe_pos = None  # novel

                images_dict = {}
                for imgnum in ["left", "right"]:
                    imgcat = r.encode_1_cat if imgnum == "left" else r.encode_2_cat
                    cue_status = ("uncued", "cued")[r.cue_position == imgnum]

                    if imgnum == probe_pos:
                        img = select_stim(
                            run_timesprobed_counter,
                            current_run_stims,
                            run_stims_dict,
                            useprobe=useprobe,
                            oper=r.operation,
                            cuestatus=cue_status,
                            imgcat=imgcat,
                            reverse=False
                        )
                    else:
                        img = select_stim(
                            run_timesprobed_counter,
                            current_run_stims,
                            run_stims_dict,
                            useprobe=useprobe,
                            oper=r.operation,
                            cuestatus=cue_status,
                            imgcat=imgcat,
                            reverse=True
                        )
                    images_dict[imgnum] = img

                # probe image
                if probe_pos is not None:
                    probe_img = images_dict[probe_pos]
                    run_timesprobed_counter[probe_img] += 1
                else:
                    probe_cat = np.random.choice([r.encode_1_cat, r.encode_2_cat])
                    probe_img = run_novel_stims_dict[probe_cat].pop()

                images_dict["probe"] = probe_img
                run_image_list.append(images_dict)

            # check this run: no None anywhere
            run_check = check_trials(run_image_list, run_len)
            t += 1
            if t > 5:
                useprobe = False  # relax balancing

        if not run_check:
            # this run never succeeded -> abort this whole build attempt
            build_ok = False
            break

        # commit run to global state
        image_list += run_image_list
        timesprobed_counter = run_timesprobed_counter
        stims_dict          = run_stims_dict
        novel_stims_dict    = run_novel_stims_dict

    # after all runs
    if build_ok and len(image_list) == total_trials:
        print("completed build, now checking...")
        check = check_trials(image_list, total_trials)
    else:
        print(f"DNF, restarting... (build_ok={build_ok}, n_trials={len(image_list)})")
        check = False

    retries += 1

if not check:
    raise RuntimeError("Could not build a complete image_list without missing images after multiple retries.")

# ------------ ATTACH IMAGES TO mainTask_df AND SANITY CHECK ------------

# Start from conditions_df and add images
mainTask_df = conditions_df.copy()
mainTask_df["encode_1_img"] = [i["left"]  for i in image_list]
mainTask_df["encode_2_img"] = [i["right"] for i in image_list]
mainTask_df["probe_img"]    = [i["probe"] for i in image_list]

# HARD SANITY CHECK: no missing images in any encoded/probe slot
if mainTask_df[["encode_1_img", "encode_2_img", "probe_img"]].isnull().any().any():
    raise ValueError("Stim list has missing images in encode_1_img / encode_2_img / probe_img – not saving.")

# Optional extra checks
assert len(mainTask_df) == total_trials
assert len(image_list) == total_trials



# Fallback or Attach Chosen Images
* Removed the replacement portion
* Need to make new premade stim lists that do not contain replacement category
* This was edited heavily, basically I saw that it kept failing after 20 tries and the same premade stim list was generate, therefore when the main_stim_list was made, it still had replace, lure, etc. Because the mirroring portion of the code was changed and balanced, I changed this code to make sure that the images are correctly added and balanced. 

The code first tries to assign specific images to every trial under all the balancing rules; if that works, it writes the chosen filenames into the trial table (encode_1_img, encode_2_img, replace_img, probe_img). But if the builder keeps failing—after more than 20 attempts—it stops trying and loads a premade, known-good stimulus list from stimuli/csvs/maintask_stimlists/. Either way, you end up with a complete, runnable list, so tight constraints or missing data don’t leave you empty-handed.

If building keeps failing after many tries, grab a pre-made, known-good list from disk. Otherwise, attach the chosen picture filenames to the trial table.
- If repeated failures occur, load a premade stimulus list from disk.
- Otherwise, insert the selected filenames (`encode_1_img`, `encode_2_img`, `replace_img`, `probe_img`) into the main DataFrame.

If retries > 20, give up and load a premade stimulus list from stimuli/csvs/maintask_stimlists/.
Otherwise, attach the selected image filenames (encode_1_img, encode_2_img, replace_img, probe_img) to mainTask_df.

Ensures the pipeline always yields a runnable list, even under extreme constraints or missing data quirks.


In [ ]:
max_retries = 20
retries = 0
success = False

while not success and retries <= max_retries:
    try:
        # (Re)generate your image_list here if needed
        # image_list = ...

        # Start from conditions_df and add images
        mainTask_df = conditions_df.copy()
        mainTask_df["encode_1_img"] = [i["left"]  for i in image_list]
        mainTask_df["encode_2_img"] = [i["right"] for i in image_list]
        mainTask_df["probe_img"]    = [i["probe"] for i in image_list]

# HARD SANITY CHECK: no missing images in any encoded/probe slot
if mainTask_df[["encode_1_img", "encode_2_img", "probe_img"]].isnull().any().any():
    raise ValueError("Stim list has missing images in encode_1_img / encode_2_img / probe_img – not saving.")

        # Put any sanity checks here; raise if something is wrong
        assert len(mainTask_df) == total_trials
        assert len(image_list) == total_trials

        success = True  # got a good list, stop retrying

    except AssertionError:
        retries += 1

# If it still failed after 20 tries, fall back to premade list
if not success:
    premade_infile = f"stimuli/csvs/maintask_stimlists/main_stim_list_{random.randint(0,9)}.csv"
    print(f"using pre-made stim list {os.path.split(premade_infile)[1]}")
    mainTask_df = pd.read_csv(premade_infile)



# Save Final CSV
* Doesn't need to be modified

Writes the final 192-trial design matrix to the subject’s `main_task` folder and stores the path in `stim_list`.

Saves the final 192-row trial table to subject lists/sub-<participant>/main_task/main_stim_list.csv and returns the path in stim_list.
PsychoPy (or your analysis code) can directly load this CSV to run the task or check balance.
That CSV is what PsychoPy (or you) will load to actually run the experiment.

In [ ]:

mainTask_df_infile = f"{list_foldername}/main_task/main_stim_list.csv"

# Save to maintask folder
mainTask_df.to_csv(mainTask_df_infile, index=False)

stim_list = mainTask_df_infile
stim_list


In [ ]:
# ------------ BUILD IMAGE LISTS SAFELY (NO MISSING IMAGES) WITH FALLBACK ------------
# EDITED VERSION: avoids dead-ends by tracking "used images" WITHIN CATEGORY per run

from collections import Counter
import copy
import numpy as np
import random

build_succeeded = False
max_retries = 20

for attempt in range(max_retries):
    # Assign stimulus pools (fresh each attempt)
    stims_dict = {}
    for oper in ["maintain", "suppress"]:
        stims_dict[oper] = {}
        for probe in ["cued", "uncued"]:
            stims_dict[oper][probe] = {
                "faces":  np.random.permutation(faces_mainTask_list).tolist(),
                "places": np.random.permutation(places_mainTask_list).tolist(),
                "fruits": np.random.permutation(fruits_mainTask_list).tolist(),
            }

    # Novel probe pools
    novel_stims_dict = {
        "faces":  np.random.permutation(faces_novel_list).tolist(),
        "places": np.random.permutation(places_novel_list).tolist(),
        "fruits": np.random.permutation(fruits_novel_list).tolist(),
    }

    # Counter: how many times each main-task image has been used as a probe (across whole task)
    timesprobed_counter = Counter(faces_mainTask_list + places_mainTask_list + fruits_mainTask_list)
    for entry in list(timesprobed_counter.keys()):
        timesprobed_counter[entry] = 0

    image_list = []   # all trials across runs
    build_ok = True   # flips False if any run fails

    # -------- per-run build --------
    for run in range(1, num_runs + 1):
        df = mainTask_df[mainTask_df["run_num"] == run]

        run_check = False
        t = 0
        useprobe = True  # prefer strict balancing at first

        while not run_check and t < 10:
            random.seed()
            run_image_list = []      # per-trial images for this run

            # EDIT: track used images WITHIN EACH CATEGORY for this run
            current_run_stims = {"faces": [], "places": [], "fruits": []}

            # local copies so failures don't corrupt global state
            run_timesprobed_counter = copy.deepcopy(timesprobed_counter)
            run_stims_dict        = copy.deepcopy(stims_dict)
            run_novel_stims_dict  = copy.deepcopy(novel_stims_dict)

            # iterate trials for this run
            for _, r in df.iterrows():
                # decide probe position
                if r.probe_subtype == "cued":
                    probe_pos = r.cue_position
                elif r.probe_subtype == "uncued":
                    probe_pos = "left" if r.cue_position == "right" else "right"
                else:
                    probe_pos = None  # novel

                images_dict = {}
                for imgnum in ["left", "right"]:
                    imgcat = r.encode_1_cat if imgnum == "left" else r.encode_2_cat
                    cue_status = ("uncued", "cued")[r.cue_position == imgnum]

                    # EDIT: pass category-specific used list
                    used_list = current_run_stims[imgcat]

                    if imgnum == probe_pos:
                        img = select_stim(
                            run_timesprobed_counter,
                            used_list,
                            run_stims_dict,
                            useprobe=useprobe,
                            oper=r.operation,
                            cuestatus=cue_status,
                            imgcat=imgcat,
                            reverse=False
                        )
                    else:
                        img = select_stim(
                            run_timesprobed_counter,
                            used_list,
                            run_stims_dict,
                            useprobe=useprobe,
                            oper=r.operation,
                            cuestatus=cue_status,
                            imgcat=imgcat,
                            reverse=True
                        )
                    images_dict[imgnum] = img

                # probe image
                if probe_pos is not None:
                    probe_img = images_dict[probe_pos]
                    run_timesprobed_counter[probe_img] += 1
                else:
                    probe_cat = np.random.choice([r.encode_1_cat, r.encode_2_cat])
                    probe_img = run_novel_stims_dict[probe_cat].pop()

                images_dict["probe"] = probe_img
                run_image_list.append(images_dict)

            # check this run: no None anywhere
            run_check = check_trials(run_image_list, run_len)
            t += 1
            if t > 5:
                useprobe = False  # relax balancing

        if not run_check:
            # this run never succeeded -> abort this whole build attempt
            build_ok = False
            break

        # commit run to global state
        image_list += run_image_list
        timesprobed_counter = run_timesprobed_counter
        stims_dict          = run_stims_dict
        novel_stims_dict    = run_novel_stims_dict

    # after all runs for this attempt
    if build_ok and len(image_list) == total_trials and check_trials(image_list, total_trials):
        print(f"completed build on attempt {attempt+1}, now checking... success!")
        build_succeeded = True
        break
    else:
        print(f"DNF, restarting... (build_ok={build_ok}, n_trials={len(image_list)})")

# ------------ ATTACH IMAGES TO mainTask_df ------------
if not build_succeeded:
    raise ValueError("Image build failed (build_succeeded=False).")

assert len(image_list) == len(mainTask_df)

mainTask_df["encode_1_img"] = [i["left"]  for i in image_list]
mainTask_df["encode_2_img"] = [i["right"] for i in image_list]
mainTask_df["probe_img"]    = [i["probe"] for i in image_list]

# ------------ SAVE SUBJECT-SPECIFIC LIST ------------
mainTask_df_infile = os.path.join(list_foldername, "main_task", "main_stim_list.csv")
mainTask_df.to_csv(mainTask_df_infile, index=False)
stim_list = mainTask_df_infile

In [ ]:
# --- ASSIGN CUE POSITION, BALANCED WITHIN (cat1, cat2, op) GROUPS ---

conditions_df["cue_position"] = -1
group_cols = ["encode_1_cat", "encode_2_cat", "operation"]

for _, idx in conditions_df.groupby(group_cols).groups.items():
    idx = list(idx)
    n = len(idx)             # now should be 16 trials per (pair × operation)
    assert n % 2 == 0         # ADDED guard
    cue_vals = np.array([0] * (n // 2) + [1] * (n // 2))  # now 8 left, 8 right per group
    np.random.shuffle(cue_vals)
    conditions_df.loc[idx, "cue_position"] = cue_vals

# --- RANDOMIZE TRIAL ORDER ---
conditions_df = conditions_df.sample(frac=1).reset_index(drop=True)

# --- ADD RUN NUMBERS: 4 runs × 48 trials ---
conditions_df["run_num"] = np.repeat(np.arange(1, num_runs + 1), run_len)  # now 1..4, each 48 trials

conditions_df.head()


## Pseudorandomize trials within runs and append to mainTask dataframe
mainTask_df = pd.DataFrame()

for i in range(1, num_runs+1):
    run_df = conditions_df[conditions_df["run_num"] == i]
    wm_valid_sequence = False
    while not wm_valid_sequence:
        run_df = run_df.sample(frac=1).reset_index(drop=True)
        wm_valid_sequence = not any(
            (run_df['operation'] == run_df['operation'].shift(1)) &
            (run_df['operation'] == run_df['operation'].shift(2)) &
            (run_df['operation'] == run_df['operation'].shift(3))
        )
    mainTask_df = pd.concat((mainTask_df, run_df), ignore_index=True)

mainTask_df.head()

# --- CHECK: overall operation counts (maintain vs suppress) ---
op_counts = mainTask_df["operation"].value_counts().reindex(["maintain", "suppress"], fill_value=0)
print("\nOverall operation counts:")
print(op_counts)

## recode the conditions into variable names used in PsychoPy
mainTask_df["cue_position"] = ["right" if cp else "left" for cp in mainTask_df.cue_position]

# Convert the probe_type to int
mainTask_df["probe_type"] = mainTask_df["probe_type"].astype(int)

# Probe subtype mapping per operation
maintain_probes = ["cued", "uncued", "novel_samecatuncued", "novel_samecatcued"]
suppress_probes = ["cued", "uncued", "novel_samecatuncued", "novel_samecatcued"]

probe_subtype = []
for _, r in mainTask_df.iterrows():
    if r.operation == "maintain":
        probe_subtype.append(maintain_probes[r.probe_type])
    elif r.operation == "suppress":
        probe_subtype.append(suppress_probes[r.probe_type])
mainTask_df["probe_subtype"] = probe_subtype

# Trial number within run (1..48), repeated for 4 runs
mainTask_df["trial_num"] = list(range(1, run_len + 1)) * num_runs  

# Jitter: per run, balanced [3,4,5] shuffled
jitter = []
for _ in range(num_runs):
    jitter += np.random.permutation([3, 4, 5] * int(run_len / 3)).tolist()
mainTask_df["jitter"] = jitter

# Rest trigger: 1 on last trial of each run, except last trial of the whole task forced to 0
mainTask_df["rest_trigger"] = [
    int(r.trial_num == run_len) if i != (total_trials - 1) else 0
    for i, r in mainTask_df.iterrows()
]

# mainTask_df.head()
# print(mainTask_df)
import pprint
pprint.pp(np.unique(mainTask_df.groupby(["encode_1_cat", "encode_2_cat", "operation", "probe_type", "cue_position"]).count()["trial_num"]))

In [ ]:
def select_stim(probecounter, crs_list, stimsdict, useprobe=True,
                oper=None, cuestatus=None, imgcat=None, reverse=False,):
    """
    Select a concrete stimulus filename from the available pool, honoring:
    - balancing of how often items have been probed (probecounter),
    - avoiding reuse within the current run (crs_list),
    - pulling from the correct pool stimsdict[operation][cue_status][category],
    - optionally prioritizing least-probed items for the probed position (useprobe=True).
    If reverse=True, invert ordering to consume more-used items for non-probed slots.
    """
    if oper is not None and cuestatus is not None and imgcat is not None:
        pool = stimsdict[oper][cuestatus][imgcat]
        if len(pool) < 5:  # threshold; tweak if you want
            if imgcat == "faces":
                refill = np.random.permutation(faces_mainTask_list).tolist()
            elif imgcat == "places":
                refill = np.random.permutation(places_mainTask_list).tolist()
            elif imgcat == "fruits":
                refill = np.random.permutation(fruits_mainTask_list).tolist()
            else:
                refill = []

            # add only items not already in the pool to avoid giant duplicates
            existing = set(pool)
            pool.extend([s for s in refill if s not in existing])
            stimsdict[oper][cuestatus][imgcat] = pool
            
    counter = Counter()
    for o in stimsdict:
        for p in stimsdict[o]:
            for c in stimsdict[o][p]:
                counter.update(stimsdict[o][p][c])

    viable_list = [sv for sv in counter.most_common() if sv[0] in stimsdict[oper][cuestatus][imgcat]]

    for value in np.unique([v for _, v in viable_list]):
        stims = [s[0] for s in viable_list if s[1] == value]

        if useprobe:
            sortedstims = [k for k in sorted(probecounter.items(), key=lambda x:x[1]) if k[0] in stims]
            if reverse:
                sortedstims.reverse()
        else:
            sortedstims = np.random.permutation(stims)

        if useprobe:
            for val in np.unique([k[1] for k in sortedstims]):
                substims = np.random.permutation([k[0] for k in sortedstims if k[1] == val])
                for s in substims:
                    if s not in crs_list:
                        stimsdict[oper][cuestatus][imgcat].remove(s)
                        crs_list.append(s)
                        return s
        else:
            for s in sortedstims:
                if s not in crs_list:
                    stimsdict[oper][cuestatus][imgcat].remove(s)
                    crs_list.append(s)
                    return s
    return None

def check_trials(imglist: list, numtrials: int):
    """
    Ensure each trial in imglist has non-None images for:
    'left', 'right', and 'probe'.
    """
    return (len([i["left"]        for i in imglist if i["left"]        is not None]) == numtrials and
            len([i["right"]       for i in imglist if i["right"]       is not None]) == numtrials and
            len([i["probe"]       for i in imglist if i["probe"]       is not None]) == numtrials)

# Initiate control flags and operation cue images
check = False
retries = 0
opcue_dict = {
    "maintain": "stimuli/cues/maintain_cue_image.png",
    "suppress": "stimuli/cues/suppress_cue_image.png",
}

# ------------ BUILD IMAGE LISTS SAFELY (NO MISSING IMAGES) WITH FALLBACK ------------

build_succeeded = False
max_retries = 20

for attempt in range(max_retries):
    # Assign stimulus pools (fresh each attempt)
    stims_dict = {}
    for oper in ["maintain", "suppress"]:
        stims_dict[oper] = {}
        for probe in ["cued", "uncued"]:
            stims_dict[oper][probe] = {
                "faces":  np.random.permutation(faces_mainTask_list).tolist(),
                "places": np.random.permutation(places_mainTask_list).tolist(),
                "fruits": np.random.permutation(fruits_mainTask_list).tolist(),
            }

    # Novel probe pools
    novel_stims_dict = {
        "faces":  np.random.permutation(faces_novel_list).tolist(),
        "places": np.random.permutation(places_novel_list).tolist(),
        "fruits": np.random.permutation(fruits_novel_list).tolist(),
    }

    # Counter: how many times each main-task image has been used as a probe
    timesprobed_counter = Counter(faces_mainTask_list + places_mainTask_list + fruits_mainTask_list)
    for entry in timesprobed_counter:
        timesprobed_counter[entry] = 0

    image_list = []   # all trials across runs
    build_ok = True   # will flip to False if any run fails

    # -------- per-run build --------
    for run in range(1, num_runs + 1):
        df = mainTask_df[mainTask_df["run_num"] == run]
        run_check = False
        t = 0
        useprobe = True  # prefer strict balancing at first

        while not run_check and t < 10:
            random.seed()
            run_image_list = []      # per-trial images for this run
            current_run_stims = []   # track images used in this run (avoid repeats)

            # local copies so failures don't corrupt global state
            run_timesprobed_counter = copy.deepcopy(timesprobed_counter)
            run_stims_dict        = copy.deepcopy(stims_dict)
            run_novel_stims_dict  = copy.deepcopy(novel_stims_dict)

            # iterate trials for this run
            for _, r in df.iterrows():
                # decide probe position
                if r.probe_subtype == "cued":
                    probe_pos = r.cue_position
                elif r.probe_subtype == "uncued":
                    probe_pos = "left" if r.cue_position == "right" else "right"
                else:
                    probe_pos = None  # novel

                images_dict = {}
                for imgnum in ["left", "right"]:
                    imgcat = r.encode_1_cat if imgnum == "left" else r.encode_2_cat
                    cue_status = ("uncued", "cued")[r.cue_position == imgnum]

                    if imgnum == probe_pos:
                        img = select_stim(
                            run_timesprobed_counter,
                            current_run_stims,
                            run_stims_dict,
                            useprobe=useprobe,
                            oper=r.operation,
                            cuestatus=cue_status,
                            imgcat=imgcat,
                            reverse=False
                        )
                    else:
                        img = select_stim(
                            run_timesprobed_counter,
                            current_run_stims,
                            run_stims_dict,
                            useprobe=useprobe,
                            oper=r.operation,
                            cuestatus=cue_status,
                            imgcat=imgcat,
                            reverse=True
                        )
                    images_dict[imgnum] = img

                # probe image
                if probe_pos is not None:
                    probe_img = images_dict[probe_pos]
                    run_timesprobed_counter[probe_img] += 1
                else:
                    probe_cat = np.random.choice([r.encode_1_cat, r.encode_2_cat])
                    probe_img = run_novel_stims_dict[probe_cat].pop()

                images_dict["probe"] = probe_img
                run_image_list.append(images_dict)

            # check this run: no None anywhere
            run_check = check_trials(run_image_list, run_len)
            t += 1
            if t > 5:
                useprobe = False  # relax balancing

        if not run_check:
            # this run never succeeded -> abort this whole build attempt
            build_ok = False
            break

        # commit run to global state
        image_list += run_image_list
        timesprobed_counter = run_timesprobed_counter
        stims_dict          = run_stims_dict
        novel_stims_dict    = run_novel_stims_dict

    # after all runs for this attempt
    if build_ok and len(image_list) == total_trials and check_trials(image_list, total_trials):
        print(f"completed build on attempt {attempt+1}, now checking... success!")
        build_succeeded = True
        break
    else:
        print(f"DNF, restarting... (build_ok={build_ok}, n_trials={len(image_list)})")

# ------------ USE BUILT LIST *OR* FALL BACK TO PREMADE ------------

if build_succeeded:
    # Attach images to the existing mainTask_df (which already has trial_num, jitter, rest_trigger)
    mainTask_df["encode_1_img"] = [i["left"]  for i in image_list]
    mainTask_df["encode_2_img"] = [i["right"] for i in image_list]
    mainTask_df["probe_img"]    = [i["probe"] for i in image_list]
    print(mainTask_df["operation"].value_counts())

    # HARD SANITY CHECK: no missing images in any encoded/probe slot
    if mainTask_df[["encode_1_img", "encode_2_img", "probe_img"]].isnull().any().any():
        raise ValueError("Stim list has missing images in encode_1_img / encode_2_img / probe_img – this should not happen.")
else:
    # FALLBACK: use an existing premade list
    premade_infile = os.path.join(
        _thisDir,
        "stimuli", "csvs", "maintask_stimlists",
        f"main_stim_list_{random.randint(0,9)}.csv"
    )
    print(f"Using pre-made stim list {os.path.split(premade_infile)[1]} after build failure.")
    mainTask_df = pd.read_csv(premade_infile)

# Make sure run_num and trial_num exist (in case premade file doesn't have them)
if "run_num" not in mainTask_df.columns:
    mainTask_df["run_num"] = np.repeat(np.arange(1, num_runs + 1), run_len)

if "trial_num" not in mainTask_df.columns:
    mainTask_df["trial_num"] = list(range(1, total_trials + 1))

# ------------ SAVE SUBJECT-SPECIFIC LIST ------------

# Convert numeric cue_position (0/1) to strings for PsychoPy
mainTask_df["cue_position"] = mainTask_df["cue_position"].replace({0: "left", 1: "right"})

mainTask_df_infile = os.path.join(list_foldername, "main_task", "main_stim_list.csv")
mainTask_df.to_csv(mainTask_df_infile, index=False)
stim_list = mainTask_df_infile